# Fixed Egr1 CC Heatmap - Python Implementation
## Using pyBigWig and matplotlib (no external deepTools needed)

This generates proper heatmaps by extracting signal from bigWig files at peak regions.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Try to import pyBigWig
try:
    import pyBigWig
    print("✓ pyBigWig is installed")
except ImportError:
    print("Installing pyBigWig...")
    !pip install pyBigWig
    import pyBigWig

# Set working directory
os.chdir('/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/testing_CC_forPaper')
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
# Load peak files
print("Loading peak files...")

shared_peaks = pd.read_csv('combined_peaks.bed', sep='\t', header=None, names=['chr', 'start', 'end'])
male_biased = pd.read_csv('male_biased_overlaps.bed', sep='\t', header=None, names=['chr', 'start', 'end'])
female_biased = pd.read_csv('female_biased_overlaps.bed', sep='\t', header=None, names=['chr', 'start', 'end'])

print(f"Shared peaks: {len(shared_peaks)}")
print(f"Male-biased peaks: {len(male_biased)}")
print(f"Female-biased peaks: {len(female_biased)}")
print(f"Total: {len(shared_peaks) + len(male_biased) + len(female_biased)}")

In [ ]:
def extract_signal_from_peaks(peak_df, bw_file, window_bp=1000, bin_size=10):
    """
    Extract signal from a bigWig file for peaks centered in a window.
    
    Args:
        peak_df: DataFrame with columns ['chr', 'start', 'end']
        bw_file: Path to bigWig file
        window_bp: Flanking window on each side (bp)
        bin_size: Bin size (bp)
    
    Returns:
        Matrix of shape (n_peaks, n_bins)
    """
    bw = pyBigWig.open(bw_file)
    
    # Number of bins
    n_bins = (window_bp * 2) // bin_size
    matrix = np.zeros((len(peak_df), n_bins))
    
    print(f"Extracting signal from {bw_file}...")
    print(f"  Matrix shape: {matrix.shape}")
    print(f"  Window: -{window_bp}bp to +{window_bp}bp")
    print(f"  Bin size: {bin_size}bp")
    
    for i, row in peak_df.iterrows():
        chrom = row['chr']
        peak_start = row['start']
        peak_end = row['end']
        peak_center = (peak_start + peak_end) // 2
        
        # Get signal for each bin
        for j in range(n_bins):
            bin_start = peak_center - window_bp + (j * bin_size)
            bin_end = bin_start + bin_size
            
            try:
                # Get mean signal in this bin
                stats = bw.stats(chrom, bin_start, bin_end, type='mean')
                if stats and stats[0] is not None:
                    matrix[i, j] = stats[0]
                else:
                    matrix[i, j] = 0.0
            except:
                matrix[i, j] = 0.0
        
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(peak_df)} peaks")
    
    bw.close()
    return matrix

# Extract signal for both samples
print("\n" + "="*80)
print("EXTRACTING MALE EGRI1 CC SIGNAL")
print("="*80)
male_shared = extract_signal_from_peaks(shared_peaks, '../cpm_bigwigs/Male_Egr1_CPM.bw')
male_male_biased = extract_signal_from_peaks(male_biased, '../cpm_bigwigs/Male_Egr1_CPM.bw')
male_female_biased = extract_signal_from_peaks(female_biased, '../cpm_bigwigs/Male_Egr1_CPM.bw')

# Combine into one matrix
male_matrix = np.vstack([male_shared, male_male_biased, male_female_biased])
print(f"\nCombined Male matrix shape: {male_matrix.shape}")

In [ ]:
print("\n" + "="*80)
print("EXTRACTING FEMALE EGRI1 CC SIGNAL")
print("="*80)
female_shared = extract_signal_from_peaks(shared_peaks, '../cpm_bigwigs/Female_Egr1_CPM.bw')
female_male_biased = extract_signal_from_peaks(male_biased, '../cpm_bigwigs/Female_Egr1_CPM.bw')
female_female_biased = extract_signal_from_peaks(female_biased, '../cpm_bigwigs/Female_Egr1_CPM.bw')

# Combine into one matrix
female_matrix = np.vstack([female_shared, female_male_biased, female_female_biased])
print(f"\nCombined Female matrix shape: {female_matrix.shape}")

In [ ]:
# Calculate statistics
print("\n" + "="*80)
print("DATA STATISTICS")
print("="*80)

print(f"\nMale signal:")
print(f"  Min: {male_matrix.min():.4f}")
print(f"  Mean: {male_matrix.mean():.4f}")
print(f"  Median: {np.median(male_matrix):.4f}")
print(f"  Std: {male_matrix.std():.4f}")
print(f"  95th percentile: {np.percentile(male_matrix, 95):.4f}")
print(f"  99th percentile: {np.percentile(male_matrix, 99):.4f}")
print(f"  Max: {male_matrix.max():.4f}")

print(f"\nFemale signal:")
print(f"  Min: {female_matrix.min():.4f}")
print(f"  Mean: {female_matrix.mean():.4f}")
print(f"  Median: {np.median(female_matrix):.4f}")
print(f"  Std: {female_matrix.std():.4f}")
print(f"  95th percentile: {np.percentile(female_matrix, 95):.4f}")
print(f"  99th percentile: {np.percentile(female_matrix, 99):.4f}")
print(f"  Max: {female_matrix.max():.4f}")

# Determine unified z-scale
z_max = np.ceil(max(np.percentile(male_matrix, 99), 
                     np.percentile(female_matrix, 99)) * 2) / 2
print(f"\nRecommended z-scale: 0 to {z_max}")

In [ ]:
# Generate MALE heatmap
plt.figure(figsize=(16, 14), dpi=150)

# Create axes for heatmap
ax = plt.gca()

# Plot heatmap
im = ax.imshow(male_matrix, aspect='auto', cmap='Blues', vmin=0, vmax=z_max, interpolation='bilinear')

# Add region separators
region_boundaries = [0, len(shared_peaks), 
                     len(shared_peaks) + len(male_biased)]
for boundary in region_boundaries[1:-1]:
    ax.axhline(y=boundary - 0.5, color='white', linewidth=2, linestyle='--')

# Labels
ax.set_xlabel('Distance from peak center (bp)', fontsize=12, fontweight='bold')
ax.set_ylabel('Peaks', fontsize=12, fontweight='bold')
ax.set_title('Male Egr1 CC - Heatmap', fontsize=14, fontweight='bold')

# X-axis labels (in bp)
n_bins = male_matrix.shape[1]
window_bp = 1000
bin_size = 10
x_positions = np.linspace(0, n_bins - 1, 5)
x_labels = [f"{int(-(window_bp) + i * (2 * window_bp) / (n_bins - 1))}" for i in range(len(x_positions))]
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels)

# Y-axis labels for regions
region_names = ['Shared', 'Male-biased', 'Female-biased']
region_sizes = [len(shared_peaks), len(male_biased), len(female_biased)]
y_ticks = []
y_labels = []
pos = 0
for name, size in zip(region_names, region_sizes):
    y_ticks.append(pos + size // 2)
    y_labels.append(name)
    pos += size

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=11, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax, label='Signal (CPM)')

plt.tight_layout()
plt.savefig('Male_Egr1CC_heatmap_FIXED.pdf', dpi=150, bbox_inches='tight')
print("✓ Saved: Male_Egr1CC_heatmap_FIXED.pdf")
plt.close()

In [ ]:
# Generate FEMALE heatmap
plt.figure(figsize=(16, 14), dpi=150)

# Create axes for heatmap
ax = plt.gca()

# Plot heatmap
im = ax.imshow(female_matrix, aspect='auto', cmap='RdPu', vmin=0, vmax=z_max, interpolation='bilinear')

# Add region separators
region_boundaries = [0, len(shared_peaks), 
                     len(shared_peaks) + len(male_biased)]
for boundary in region_boundaries[1:-1]:
    ax.axhline(y=boundary - 0.5, color='white', linewidth=2, linestyle='--')

# Labels
ax.set_xlabel('Distance from peak center (bp)', fontsize=12, fontweight='bold')
ax.set_ylabel('Peaks', fontsize=12, fontweight='bold')
ax.set_title('Female Egr1 CC - Heatmap', fontsize=14, fontweight='bold')

# X-axis labels (in bp)
n_bins = female_matrix.shape[1]
window_bp = 1000
bin_size = 10
x_positions = np.linspace(0, n_bins - 1, 5)
x_labels = [f"{int(-(window_bp) + i * (2 * window_bp) / (n_bins - 1))}" for i in range(len(x_positions))]
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels)

# Y-axis labels for regions
region_names = ['Shared', 'Male-biased', 'Female-biased']
region_sizes = [len(shared_peaks), len(male_biased), len(female_biased)]
y_ticks = []
y_labels = []
pos = 0
for name, size in zip(region_names, region_sizes):
    y_ticks.append(pos + size // 2)
    y_labels.append(name)
    pos += size

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=11, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax, label='Signal (CPM)')

plt.tight_layout()
plt.savefig('Female_Egr1CC_heatmap_FIXED.pdf', dpi=150, bbox_inches='tight')
print("✓ Saved: Female_Egr1CC_heatmap_FIXED.pdf")
plt.close()

In [ ]:
# Generate profile plots (mean signal per region type)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=150)

# X-axis for plotting
n_bins = male_matrix.shape[1]
window_bp = 1000
bin_size = 10
x_axis = np.linspace(-window_bp, window_bp, n_bins)

# MALE profile
ax = axes[0]
pos = 0
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
region_names = ['Shared', 'Male-biased', 'Female-biased']
region_sizes = [len(shared_peaks), len(male_biased), len(female_biased)]

for color, name, size in zip(colors, region_names, region_sizes):
    region_matrix = male_matrix[pos:pos+size, :]
    mean_signal = region_matrix.mean(axis=0)
    std_signal = region_matrix.std(axis=0)
    
    ax.plot(x_axis, mean_signal, linewidth=2, label=name, color=color)
    ax.fill_between(x_axis, mean_signal - std_signal, mean_signal + std_signal, 
                     alpha=0.2, color=color)
    pos += size

ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Distance from peak center (bp)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Signal (CPM)', fontsize=11, fontweight='bold')
ax.set_title('Male Egr1 CC - Profile Plot', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# FEMALE profile
ax = axes[1]
pos = 0

for color, name, size in zip(colors, region_names, region_sizes):
    region_matrix = female_matrix[pos:pos+size, :]
    mean_signal = region_matrix.mean(axis=0)
    std_signal = region_matrix.std(axis=0)
    
    ax.plot(x_axis, mean_signal, linewidth=2, label=name, color=color)
    ax.fill_between(x_axis, mean_signal - std_signal, mean_signal + std_signal, 
                     alpha=0.2, color=color)
    pos += size

ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Distance from peak center (bp)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Signal (CPM)', fontsize=11, fontweight='bold')
ax.set_title('Female Egr1 CC - Profile Plot', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Egr1CC_profile_comparison_FIXED.pdf', dpi=150, bbox_inches='tight')
print("✓ Saved: Egr1CC_profile_comparison_FIXED.pdf")
plt.close()

In [ ]:
# Create a combined figure matching the paper output
fig = plt.figure(figsize=(18, 12), dpi=150)
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Top row: Line plots
ax_male_line = fig.add_subplot(gs[0, 0])
ax_female_line = fig.add_subplot(gs[0, 1])

# Bottom row: Heatmaps
ax_male_heat = fig.add_subplot(gs[1, 0])
ax_female_heat = fig.add_subplot(gs[1, 1])

# X-axis for line plots
n_bins = male_matrix.shape[1]
window_bp = 1000
x_axis = np.linspace(-window_bp, window_bp, n_bins)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
region_names = ['Shared', 'Male-biased', 'Female-biased']
region_sizes = [len(shared_peaks), len(male_biased), len(female_biased)]

# MALE LINE PLOT
pos = 0
for color, name, size in zip(colors, region_names, region_sizes):
    region_matrix = male_matrix[pos:pos+size, :]
    mean_signal = region_matrix.mean(axis=0)
    std_signal = region_matrix.std(axis=0)
    ax_male_line.plot(x_axis, mean_signal, linewidth=2.5, label=name, color=color)
    ax_male_line.fill_between(x_axis, mean_signal - std_signal, mean_signal + std_signal, alpha=0.2, color=color)
    pos += size

ax_male_line.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
ax_male_line.set_xlabel('Distance from peak center (bp)', fontsize=11, fontweight='bold')
ax_male_line.set_ylabel('Signal (CPM)', fontsize=11, fontweight='bold')
ax_male_line.set_title('Male Egr1 CC', fontsize=12, fontweight='bold')
ax_male_line.legend(fontsize=10, loc='upper right')
ax_male_line.grid(True, alpha=0.3)

# FEMALE LINE PLOT
pos = 0
for color, name, size in zip(colors, region_names, region_sizes):
    region_matrix = female_matrix[pos:pos+size, :]
    mean_signal = region_matrix.mean(axis=0)
    std_signal = region_matrix.std(axis=0)
    ax_female_line.plot(x_axis, mean_signal, linewidth=2.5, label=name, color=color)
    ax_female_line.fill_between(x_axis, mean_signal - std_signal, mean_signal + std_signal, alpha=0.2, color=color)
    pos += size

ax_female_line.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
ax_female_line.set_xlabel('Distance from peak center (bp)', fontsize=11, fontweight='bold')
ax_female_line.set_ylabel('Signal (CPM)', fontsize=11, fontweight='bold')
ax_female_line.set_title('Female Egr1 CC', fontsize=12, fontweight='bold')
ax_female_line.legend(fontsize=10, loc='upper right')
ax_female_line.grid(True, alpha=0.3)

# MALE HEATMAP
im_male = ax_male_heat.imshow(male_matrix, aspect='auto', cmap='Blues', vmin=0, vmax=z_max, interpolation='bilinear')
region_boundaries = [0, len(shared_peaks), len(shared_peaks) + len(male_biased)]
for boundary in region_boundaries[1:-1]:
    ax_male_heat.axhline(y=boundary - 0.5, color='white', linewidth=2, linestyle='--')

ax_male_heat.set_xlabel('Distance from peak center (bp)', fontsize=10, fontweight='bold')
ax_male_heat.set_ylabel('Peaks', fontsize=10, fontweight='bold')
n_bins = male_matrix.shape[1]
x_positions = np.linspace(0, n_bins - 1, 5)
x_labels = [f"{int(-(window_bp) + i * (2 * window_bp) / (n_bins - 1))}" for i in range(len(x_positions))]
ax_male_heat.set_xticks(x_positions)
ax_male_heat.set_xticklabels(x_labels)
y_ticks = []
y_labels = []
pos = 0
for name, size in zip(region_names, region_sizes):
    y_ticks.append(pos + size // 2)
    y_labels.append(name)
    pos += size
ax_male_heat.set_yticks(y_ticks)
ax_male_heat.set_yticklabels(y_labels, fontsize=9, fontweight='bold')
cbar_male = plt.colorbar(im_male, ax=ax_male_heat, label='Signal (CPM)')

# FEMALE HEATMAP
im_female = ax_female_heat.imshow(female_matrix, aspect='auto', cmap='RdPu', vmin=0, vmax=z_max, interpolation='bilinear')
for boundary in region_boundaries[1:-1]:
    ax_female_heat.axhline(y=boundary - 0.5, color='white', linewidth=2, linestyle='--')

ax_female_heat.set_xlabel('Distance from peak center (bp)', fontsize=10, fontweight='bold')
ax_female_heat.set_ylabel('Peaks', fontsize=10, fontweight='bold')
ax_female_heat.set_xticks(x_positions)
ax_female_heat.set_xticklabels(x_labels)
ax_female_heat.set_yticks(y_ticks)
ax_female_heat.set_yticklabels(y_labels, fontsize=9, fontweight='bold')
cbar_female = plt.colorbar(im_female, ax=ax_female_heat, label='Signal (CPM)')

fig.suptitle('Egr1 CC Sex-Biased Peak Analysis', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('Egr1CC_comprehensive_analysis_FIXED.pdf', dpi=150, bbox_inches='tight')
print("✓ Saved: Egr1CC_comprehensive_analysis_FIXED.pdf")
plt.close()

In [ ]:
# Verify all outputs were created
print("\n" + "="*80)
print("VERIFICATION OF OUTPUT FILES")
print("="*80)

output_files = [
    'Male_Egr1CC_heatmap_FIXED.pdf',
    'Female_Egr1CC_heatmap_FIXED.pdf',
    'Egr1CC_profile_comparison_FIXED.pdf',
    'Egr1CC_comprehensive_analysis_FIXED.pdf'
]

for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024  # Convert to MB
        print(f"✓ {f} ({size:.2f} MB)")
    else:
        print(f"✗ {f} (NOT FOUND)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)